### Importing Pandas

In [1]:
import pandas as pd

## Stage 1 : Data Ingestion using pandas

In [ ]:

df_deliveries = pd.read_csv("deliveries.csv")
df_matches = pd.read_csv("matches.csv")

In [ ]:
print("Deliveries : \n",df_deliveries.head()) # deliveries.csv
print("\n\nMatches : \n",df_matches.head()) # matches.csv

In [ ]:
# shapes
print("Deliveries shape : ",df_deliveries.shape)
print("Matches shape : ", df_matches.shape)

# columns
print("\nDeliveries columns : \n",df_deliveries.columns)
print("\n")
print("Matches columns :\n",df_matches.columns)

# data types
print("\n Deliveries data types : ",df_deliveries.dtypes)
print("\nMatches data types : ",df_matches.dtypes)

## Stage 2 : Data Cleaning & Validation 

In [ ]:
# checking number of null values in both files

df_deliveries.isnull().sum()
# df_matches.isnull().sum()

In [69]:
# handling null values in deliveries.csv

#changing the NA values in player_dismissed, dismissal_kind, fielder to 'none'
df_deliveries = df_deliveries.fillna('none')

In [ ]:
# Removing inconsitency from deliveries.csv

# checking unique values foir inconsistency
df_deliveries['batting_team'].unique()
df_deliveries['bowling_team'].unique()

In [ ]:

# setting the correct batting team name
df_deliveries['batting_team'] = df_deliveries['batting_team'].replace({
    'Royal Challengers Bengaluru': 'Royal Challengers Bangalore',
    'Rising Pune Supergiant' : 'Rising Pune Supergiants',
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings'
})

# setting the correct bowling team name
df_deliveries['bowling_team'] = df_deliveries['bowling_team'].replace({
    'Royal Challengers Bengaluru': 'Royal Challengers Bangalore',
    'Rising Pune Supergiant' : 'Rising Pune Supergiants',
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings'
})

# checking if total runs are valid
df_deliveries[df_deliveries['total_runs'] != df_deliveries['batsman_runs'] + df_deliveries['extra_runs']]


In [71]:
# handling null values in matches.csv

# null cities
stadium_to_city = {
    'Dubai International Cricket Stadium': 'Dubai',
    'Sharjah Cricket Stadium': 'Sharjah'
}
 
df_matches['city'] = df_matches['city'].fillna(df_matches['venue'].map(stadium_to_city))


# null values in winner and player_of_match 
df_matches['winner'] = df_matches['winner'].fillna('no_result')
df_matches['player_of_match'] = df_matches['player_of_match'].fillna('none')

# null values in result_margin : changing to 0 (tie or abandoned match)
df_matches['result_margin'] = df_matches['result_margin'].fillna(0)

#  null values in method column : NA changes to 'normal'
df_matches['method'] = df_matches['method'].fillna('normal')

#

In [ ]:
print(df_matches['team1'].unique())

In [ ]:
# Removing inconsitency in matches.csv

# setting the correct team1 name
df_matches['team1'] = df_matches['team1'].replace({
    'Royal Challengers Bengaluru': 'Royal Challengers Bangalore',
    'Rising Pune Supergiant' : 'Rising Pune Supergiants',
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings'
})

# setting the correct team2 name
df_matches['team2'] = df_matches['team2'].replace({
    'Royal Challengers Bengaluru': 'Royal Challengers Bangalore',
    'Rising Pune Supergiant' : 'Rising Pune Supergiants',
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings'
})



In [ ]:
# Validating ids

# Check if all delivery match_ids exist in matches
invalid_ids = df_deliveries[~df_deliveries['match_id'].isin(df_matches['id'])]
len(invalid_ids)

print(invalid_ids.head())

In [ ]:
# correct data types (numeric vs categorical)
df_deliveries.info()
print('\n\n')
df_matches.info()

## Stage 3: Data Transformation

In [93]:
# Prepare data for analysis.

#Create new columns:

# total runs scored
df_deliveries['total_runs_calculated'] = df_deliveries['batsman_runs'] + df_deliveries['extra_runs']

# runs from batsman
df_deliveries['pure_batting_runs'] = df_deliveries['batsman_runs']

# total balls
df_deliveries['ball_number'] = df_deliveries['over'] * 6 + df_deliveries['ball']


print(df_deliveries['total_runs_calculated'].head())
print(df_deliveries['pure_batting_runs'].head())
print(df_deliveries['ball_number'].head())



0    1
1    0
2    1
3    0
4    0
Name: total_runs_calculated, dtype: int64
0    0
1    0
2    0
3    0
4    0
Name: pure_batting_runs, dtype: int64
0    1
1    2
2    3
3    4
4    5
Name: ball_number, dtype: int64


In [96]:
# Standardize columns

# changing column names to match
df_matches.rename(columns={'id': 'match_id'}, inplace=True)

# for avoiding confusion
df_deliveries.columns = df_deliveries.columns.str.lower()
df_matches.columns = df_matches.columns.str.lower()

print(df_matches.columns)

Index(['match_id', 'season', 'city', 'date', 'match_type', 'player_of_match',
       'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner',
       'result', 'result_margin', 'target_runs', 'target_overs', 'super_over',
       'method', 'umpire1', 'umpire2'],
      dtype='str')


In [ ]:
# merge
df_final = df_deliveries.merge(
    df_matches,
    on='match_id',
    how='left'
)

print(df_deliveries.shape)
print(df_matches.shape)
print(df_final.shape)

(260920, 20)
(260920, 39)
